In [28]:
import torch
from torch import nn
from d2l import torch as d2l

In [29]:
# NiN Block Architecture

def nin_block(
    out_channels: int,
    kernel_size: int,
    stride: int,
    padding: int,
) -> nn.Sequential:
    
    return nn.Sequential(
        
        # Spatial feature 추출
        nn.LazyConv2d(
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        ),
        nn.ReLU(),
        
        # 각 pixel의 channel vector에 대해
        # 1x1 Convolution (Linear transformation) 적용
        nn.LazyConv2d(
            out_channels=out_channels,
            kernel_size=1,
        ),
        nn.ReLU(),
        
        nn.LazyConv2d(
            out_channels=out_channels,
            kernel_size=1,
        ),
        nn.ReLU(),
    )

In [30]:
# Tracing NiN Block Tensor Shape

# [B, in, H, W]
X = torch.randn(
    1,
    3,
    8,
    8,
)

# [B, out, H, W]
block = nin_block(
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1,
)

with torch.no_grad():
    for layer in block:
        X = layer(X)

        print(
            f"{layer.__class__.__name__:<12}",
            "output shape:",
            tuple(X.shape),
        )

Conv2d       output shape: (1, 16, 8, 8)
ReLU         output shape: (1, 16, 8, 8)
Conv2d       output shape: (1, 16, 8, 8)
ReLU         output shape: (1, 16, 8, 8)
Conv2d       output shape: (1, 16, 8, 8)
ReLU         output shape: (1, 16, 8, 8)


In [35]:
# 1x1 Convolution과 Linear Transformation 비교

# [B, C, H, W]
X_small = torch.randn(
    1,
    3,
    2,
    2,
)

# [1, 2, 2, 2]
pointwise_conv = nn.Conv2d(
    in_channels=3,
    out_channels=2,
    kernel_size=1,
)

bias = pointwise_conv.bias

with torch.no_grad():
    
    # Pytorch: 1x1 Convolution Layer Forward
    convolution_output = pointwise_conv(
        X_small
    )
    
    # "왼쪽 위 pixel"의 input_channel vector
    pixel_vector = X_small[
        0,
        :,
        0,
        0,
    ] # Column Vector: [R, G, B].T


    # Kernel(weight) tensor shape: [out, in, H=1, W=1]
    # 1x1 Conv의 weight를 matrix로 변환 -> [out, in]
    weight_matrix = pointwise_conv.weight[
        :,
        :,
        0,
        0,
    ]
    
    # [out, in] @ [in] = [out, 1]
    linear_output = (
        weight_matrix @ pixel_vector
        + bias
    )
    
    
print("Input X_small:", X_small.shape)
print(X_small)

print("\n왼쪽 위 pixel_vector:", pixel_vector.shape)
print(pixel_vector)

print("\nPyTorch써서 나온 1x1 convolution_output.shape:", convolution_output.shape)
print(convolution_output)

print("\npointwise_conv 네트워크의 weight", pointwise_conv.weight.shape)
print(pointwise_conv.weight)


print(
    "\nPyTorch 써서 나온 왼쪽 위 1x1 Conv output:",
    convolution_output[0, :, 0, 0],
)
print(
    "\nLinear output:",
    linear_output,
)
print(
    "Same result:",
    torch.allclose(
        convolution_output[0, :, 0, 0],
        linear_output,
    ),
)

Input X_small: torch.Size([1, 3, 2, 2])
tensor([[[[-0.1653,  0.1637],
          [-1.6119,  0.1699]],

         [[ 0.6876,  0.0591],
          [-0.2652,  0.1344]],

         [[-1.2372,  0.4034],
          [ 0.8261,  0.5732]]]])

왼쪽 위 pixel_vector: torch.Size([3])
tensor([-0.1653,  0.6876, -1.2372])

PyTorch써서 나온 1x1 convolution_output.shape: torch.Size([1, 2, 2, 2])
tensor([[[[-1.1607, -0.3528],
          [-0.1855, -0.2706]],

         [[-0.0370,  0.8002],
          [ 0.1274,  0.9145]]]])

pointwise_conv 네트워크의 weight torch.Size([2, 3, 1, 1])
Parameter containing:
tensor([[[[ 0.0225]],

         [[-0.0061]],

         [[ 0.4856]]],


        [[[ 0.4526]],

         [[ 0.2868]],

         [[ 0.5294]]]], requires_grad=True)

PyTorch 써서 나온 왼쪽 위 1x1 Conv output: tensor([-1.1607, -0.0370])

Linear output: tensor([-1.1607, -0.0370])
Same result: True
